# GoldWork_Incremental
- Incremental Gold processing plus latest and timestamped volume snapshots

# Step 1  -- Imports and Setup
This cell imports The required helpers, switches to the right catlog, makes sure the gold schema exists and creates

- a `gold_run_id`
- a run date string
- a run timestamp string

Thease are used for tracking and snapshot publishing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog venk_novacart")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())


run_ts_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.now().strftime("%Y-%m-%d")

print("current Gold Run ID",run_ts_str)
print("Run Timestamp Folder",run_date_str)

# Step -2 -- Gold control table
This table stores the latest Gold processing State

it tells Gold
- which silver data was processed last time
- how many gold rows were merged in last run

In [0]:
spark.sql("""
create table if not exists gold_schema.processing_control (
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
)
using delta
""")

# Step 3 --helper functions
This cell defines reusable Gold functions:
- upsert_to_gold() merges data into Gold current state tables
- get_last_processed_silver_ts() reads gold watermark from control table
- upsert_gold_control() updates Gold control after a sucessful run

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
           .merge(df_source.alias("source"),
                  f"target.{target_table} = source.{join_key}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.table("venk_novacart.processing_control")
        .filter(
            (F.col("layer") == "gold") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None

    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
        layer string,
        entity_name string,
        last_processed_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "gold_schema.processing_control") .alias('t') .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name") .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        }) \
        .whenNotMatchedInsertAll() \
        .execute()
        